In [ ]:
import numpy as np
import pandas as pd
from pandas import DataFrame, Series
import matplotlib.pyplot as plt

%matplotlib inline

import pims
import trackpy as tp

%load_ext autoreload
%autoreload 2

In [ ]:
# Loads spots file

path = '/Volumes/gchao/bamfaile/Analysis/TUBB2B-KI/Batch20230223/D21/Spotdetection_tracking/Spotdetection/100tp_561-100-50ms-1000g_4_conf561_merged_spots.csv'
spots = pd.read_csv(path)
spots.head()

In [ ]:
def split_table_by_roi_id(df):
    """
    Splits each spots file into seperate dfs by roi-id, so that tracking can be done per ROI 
    """
    df = df.groupby("roi_id", sort = False, as_index = False)
    
    return df

In [ ]:
spots_per_ROI = [j for i,j in split_table_by_roi_id(spots)]
spots_per_ROI[0].head(10)

In [ ]:
# Links spots for all ROIs, filters them by minimum track length (default: 4 frames)
# Concatenates results of single ROIs into one df
# First number in df name stands for the distance in pixels that two spots can be apart, second number stands for how
# many frames a spot can be missing, before it will not be assigned to the same track anymore

tracks_per_ROI_5_3 = []

for df in spots_per_ROI:
    df = tp.link(df, 5, memory = 3)
    df = tp.filter_stubs(df, 4)
    tracks_per_ROI_5_3.append(df)

#all_tracks_5_3 = pd.concat(tracks_per_ROI_5_3)

#all_tracks_5_3.head(10)

In [ ]:
all_tracks_5_3 = pd.concat(tracks_per_ROI_5_3)
all_tracks_5_3.head(10)

In [ ]:
# Takes columns "particle", "frame", "roi_id", "X", "y" and sorts into format required by napari for visualization
# Inserts channel information right before y and x coordinate, according to image dimensions

all_tracks_5_3_sorted = all_tracks_5_3.iloc[:, np.r_[9,0,3,2]].sort_values("particle")
all_tracks_5_3_sorted.insert(2, "channel", 0)
all_tracks_5_3_sorted.head(20)

In [ ]:
plt.figure()
tp.plot_traj(tracks_per_roi0_filt);